### <font color=Blue> Conversational Memory Using MemorySaver

MemoerySaver() of langchain's agent manages short-term memory as a part of agent’s state.

Let's look at an example use of InMemorySaver

    Let's develop an intelligent conversational agent capable of maintaining memory across interactions and providing real-time, personalized responses. We will use the Tavily API as a search tool for the agent to retrieve real-time information on various topics.

#### Tavily API

The Tavily Search API is a search engine built specifically for AI agents rather than humans.

    While a human uses Google to get a list of links to click on, an AI agent uses Tavily to get clean, structured, and factual data that it can immediately "read" and use to make decisions.

In [ ]:
## Import desired libraries and modules

In [2]:
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

In [10]:
## Set Tavily API key in the environment

In [3]:
import getpass
import os

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("TAVILY_API_KEY")


TAVILY_API_KEY:  ········


In [6]:
#os.environ["TAVILY_API_KEY"]

#### Create a model instance

In [7]:
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=4000)
llm.invoke("Hi").content

'Hello! How can I help you today?'

#### Creating Tavily Tool for Internet Search

In [13]:
from langchain_tavily import TavilySearch
tavily_search = TavilySearch(max_results=3,
                      search_depth="advanced",
                     )
tools = [tavily_search]


In [14]:
tavily_search.invoke("current news on cricket")

{'query': 'current news on cricket',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.skysports.com/cricket/news',
   'title': 'Cricket News - Live Scores, Highlights, Results | Sky Sports',
   'content': "Sky Sports\n\nCricket\n\n#### Internationals\n\n#### Domestic\n\n#### Indian Premier League\n\n#### Domestic Leagues\n\n#### Internationals\n\n# Cricket News\n\nEngland's Ollie Pope reacts as walks from the field after he was dismissed during play on day two of the third Ashes cricket test between England and Australia in Adelaide, Australia, Thursday, Dec. 18, 2025. (AP Photo/James Elsby)\n\n### Pope: Perception England 'weren't fussed' about Ashes was tough\n\nEngland's Liam Livingstone, ODI cricket (Associated Press)\n\n### Livingstone slams England cricket regime - 'No-one cares about you'\n\nThe Hundred auction\n\n### What challenges does The Hundred face after cash splashed in auction?\n\nJames Coles, The Hundred, Southern Brave (G

#### Create In MemorySaver instance for implementing short term memory for agents

In [9]:
memory = InMemorySaver()

### Create a LangChain Prebuilt Agent's Instance >> create_agent

In [38]:

agent = create_agent(llm,tools,checkpointer=memory, system_prompt="you are a helpful assistant")

### Configure the Agent Instance

In [39]:

config = {"configurable": {"thread_id": "Sep30"}}

### Stream run the Agent

In [40]:
from rich import print

for chunk in agent.stream(
    {"messages": [HumanMessage(content="Hi I'm Ram and I live in Mysore")]}, config
):
    print(chunk)
    print("----")

{
    'model': {
        'messages': [
            AIMessage(
                content="Hi Ram, it's nice to meet you!",
                additional_kwargs={},
                response_metadata={
                    'ResponseMetadata': {
                        'RequestId': '772ceb5b-b417-4ad3-aafd-2fed4f7ad59d',
                        'HTTPStatusCode': 200,
                        'HTTPHeaders': {
                            'date': 'Mon, 16 Mar 2026 09:49:02 GMT',
                            'content-type': 'application/json',
                            'content-length': '237',
                            'connection': 'keep-alive',
                            'x-amzn-requestid': '772ceb5b-b417-4ad3-aafd-2fed4f7ad59d'
                        },
                        'RetryAttempts': 0
                    },
                    'stopReason': 'end_turn',
                    'metrics': {'latencyMs': [3063]},
                    'model_provider': 'bedrock_converse',
                    'model_name': 'cohere.command-r-plus-v1:0'
                },
                id='lc_run--019cf60c-7d04-7d91-aa2d-565e4609ccfe-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 1406,
                    'output_tokens': 14,
                    'total_tokens': 1420,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0}
                }
            )
        ]
    }
}

----

In [41]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="Hi I'm Ram and I live in Mysore")]}, config
):
    print(chunk)
    print("----")

{
    'model': {
        'messages': [
            AIMessage(
                content="Hi Ram, it's nice to meet you! How are you today?",
                additional_kwargs={},
                response_metadata={
                    'ResponseMetadata': {
                        'RequestId': '7cf620ed-c915-4c07-8b57-eb77054d868c',
                        'HTTPStatusCode': 200,
                        'HTTPHeaders': {
                            'date': 'Mon, 16 Mar 2026 09:49:06 GMT',
                            'content-type': 'application/json',
                            'content-length': '256',
                            'connection': 'keep-alive',
                            'x-amzn-requestid': '7cf620ed-c915-4c07-8b57-eb77054d868c'
                        },
                        'RetryAttempts': 0
                    },
                    'stopReason': 'end_turn',
                    'metrics': {'latencyMs': [3595]},
                    'model_provider': 'bedrock_converse',
                    'model_name': 'cohere.command-r-plus-v1:0'
                },
                id='lc_run--019cf60c-89d6-7360-8e7f-2e50c5ef3e79-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 1431,
                    'output_tokens': 19,
                    'total_tokens': 1450,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0}
                }
            )
        ]
    }
}

----

### Extract the AI Message

In [42]:
chunk['model']['messages'][0].content

"Hi Ram, it's nice to meet you! How are you today?"

### Check for Agent's Memory 

In [43]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="What's the latest news update where I live?")]}, config
):
    print(chunk)
    print("----")

{
    'model': {
        'messages': [
            AIMessage(
                content=[
                    {'type': 'text', 'text': 'I will search for the latest news in Mysore.'},
                    {
                        'type': 'tool_use',
                        'name': 'tavily_search',
                        'input': {'query': 'latest news Mysore', 'time_range': 'day'},
                        'id': 'tooluse_FUFpzJ53EyvRmVTPdSrGxS'
                    }
                ],
                additional_kwargs={},
                response_metadata={
                    'ResponseMetadata': {
                        'RequestId': '76a848f8-c7e7-4593-a79e-11facb61471f',
                        'HTTPStatusCode': 200,
                        'HTTPHeaders': {
                            'date': 'Mon, 16 Mar 2026 09:49:08 GMT',
                            'content-type': 'application/json',
                            'content-length': '391',
                            'connection': 'keep-alive',
                            'x-amzn-requestid': '76a848f8-c7e7-4593-a79e-11facb61471f'
                        },
                        'RetryAttempts': 0
                    },
                    'stopReason': 'tool_use',
                    'metrics': {'latencyMs': [2250]},
                    'model_provider': 'bedrock_converse',
                    'model_name': 'cohere.command-r-plus-v1:0'
                },
                id='lc_run--019cf60c-98c2-7910-8f2a-0ef1627f99a5-0',
                tool_calls=[
                    {
                        'name': 'tavily_search',
                        'args': {'query': 'latest news Mysore', 'time_range': 'day'},
                        'id': 'tooluse_FUFpzJ53EyvRmVTPdSrGxS',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 1441,
                    'output_tokens': 28,
                    'total_tokens': 1469,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0}
                }
            )
        ]
    }
}

----

{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"query": "latest news Mysore", "follow_up_questions": null, "answer": null, "images": [],
"results": [{"url": 
"https://observervoice.com/rashmika-mandannas-silk-saree-sparks-a-revival-for-mysore-silk-looms-191181/", "title": 
"Rashmika Mandanna\'s Silk Saree Sparks a Revival for Mysore Silk ...", "content": "OV News DeskMarch 16, 2026Last 
Updated: March 16, 2026\\n\\n2 minutes read\\n\\nFollow Us\\n\\nShare\\n\\n Facebook   X   LinkedIn   Pinterest   
Reddit   Skype   Messenger   Messenger   WhatsApp   Telegram   Share via Email   Print\\n\\n### OV News 
Desk\\n\\nThe OV News Desk comprises a professional team of news writers and editors working round the clock to 
deliver timely updates on business, technology, policy, world affairs, sports and current events. The desk combines
editorial judgment with journalistic integrity to ensure every story is accurate, fact-checked, and relevant. From 
market… More »\\n\\n Website\\n Facebook\\n X\\n LinkedIn\\n YouTube\\n Instagram\\n\\n## Read 
Next\\n\\nEntertainment\\n\\nMarch 16, 2026\\n\\n## Rohit Shetty Unveils Golmaal 5 Making Video: Akshay Kumar Goes 
Bald and Sharman Joshi Returns After 20 Years! [...] March 10, 2026\\n\\n### Rhea Chakraborty Unveils AI Avatar 
‘Mishty’ with Collective Artists Network\\n\\nMarch 10, 2026\\n\\n### Sitaare Zameen Par Shifts from YouTube 
Rentals to SonyLIV: A New Streaming Journey\\n\\nMarch 10, 2026\\n\\n### Rohan and Vinayak Dive into the Soundscape
of Anil Kapoor’s Latest Film\\n\\nFacebook   X   LinkedIn   Pinterest   Reddit   Skype   Messenger   Messenger   
Share via Email\\n\\nFacebook   X   LinkedIn   Skype   Messenger   Messenger   WhatsApp\\n\\n Back to top 
button\\n\\nClose\\n\\nClose [...] Entertainment\\n\\nMarch 16, 2026\\n\\n## Anshuman Jha Drops Teaser Poster for 
Lakadbaggha 2: The Monkey Business, Set for Diwali 2026 Release\\n\\nEntertainment\\n\\nMarch 10, 2026\\n\\n## Rhea
Chakraborty Unveils AI Avatar ‘Mishty’ with Collective Artists Network\\n\\nEntertainment\\n\\nMarch 10, 
2026\\n\\n## Sitaare Zameen Par Shifts from YouTube Rentals to SonyLIV: A New Streaming 
Journey\\n\\nEntertainment\\n\\nMarch 10, 2026\\n\\n## Rohan and Vinayak Dive into the Soundscape of Anil Kapoor’s 
Latest Film\\n\\nMarch 16, 2026\\n\\n### Rohit Shetty Unveils Golmaal 5 Making Video: Akshay Kumar Goes Bald and 
Sharman Joshi Returns After 20 Years!\\n\\nMarch 16, 2026\\n\\n### Anshuman Jha Drops Teaser Poster for Lakadbaggha
2: The Monkey Business, Set for Diwali 2026 Release\\n\\nMarch 10, 2026", "score": 0.4767995, "raw_content": null},
{"url": 
"https://www.deccanherald.com/india/karnataka/bengaluru/residents-oppose-plan-for-convention-centre-on-mysore-lamps
-factory-premises-in-bengaluru-3933053", "title": "Mysore Lamps Site: Residents oppose convention centre plan in 
...", "content": "Sign in\\n\\nDH Specials\\n\\nIndia 
Politics\\n\\nBengaluru\\n\\nTechnology\\n\\nLifestyle\\n\\nTrending\\n\\nPhotos\\n\\nDH 
Brandspot\\n\\nMenu\\n\\n×\\n\\nADVERTISEMENT\\n\\nADVERTISEMENT\\n\\nHomeindiakarnatakabengaluru\\n\\n# Residents 
oppose plan for convention centre on Mysore Lamps factory premises in Bengaluru\\n\\nResidents of Malleswaram have 
urged the government to instead declare the land a public green space or a \'deemed forest\', citing its ecological
and historical significance.\\n\\nDHNS\\n\\nLast Updated : 15 March 2026, 21:10IST\\n\\nADVERTISEMENT\\n\\nFollow 
Us :\\n\\nComments\\n\\nADVERTISEMENT\\n\\nPublished 15 March 2026, 21:10IST\\n\\nBengaluruKarnataka News\\n\\n## 
Follow us on :\\n\\n## Follow Us", "score": 0.46482924, "raw_content": null}, {"url": 
"https://timesofindia.indiatimes.com/city/mysuru/community-reclaims-bogadi-road-junction-demands-inclusive-streets/
articleshow/129593297.cms", "title": "Community reclaims Bogadi Road Junction, demands inclusive streets", 
"content": "Article image for: ‘Because I Am Brahmin…’: Mani Shankar 

----

{
    'model': {
        'messages': [
            AIMessage(
                content="Here are some of the latest news updates from Mysore:\n- Residents of Malleswaram have 
opposed the plan for a convention centre on Mysore Lamps factory premises, instead urging the government to declare
the land a public green space or a 'deemed forest'.\n- Community members have reclaimed Bogadi Road Junction and 
are demanding inclusive streets.",
                additional_kwargs={},
                response_metadata={
                    'ResponseMetadata': {
                        'RequestId': '00b58eda-3468-4404-8957-b7910b0c303c',
                        'HTTPStatusCode': 200,
                        'HTTPHeaders': {
                            'date': 'Mon, 16 Mar 2026 09:49:18 GMT',
                            'content-type': 'application/json',
                            'content-length': '555',
                            'connection': 'keep-alive',
                            'x-amzn-requestid': '00b58eda-3468-4404-8957-b7910b0c303c'
                        },
                        'RetryAttempts': 0
                    },
                    'stopReason': 'end_turn',
                    'metrics': {'latencyMs': [6258]},
                    'model_provider': 'bedrock_converse',
                    'model_name': 'cohere.command-r-plus-v1:0'
                },
                id='lc_run--019cf60c-ae97-72f2-b8c9-682c44da5f6e-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 3206,
                    'output_tokens': 69,
                    'total_tokens': 3275,
                    'input_token_details': {'cache_creation': 0, 'cache_read': 0}
                }
            )
        ]
    }
}

----

In [44]:
print(chunk['model']['messages'][0].content)

Here are some of the latest news updates from Mysore:
- Residents of Malleswaram have opposed the plan for a convention centre on Mysore Lamps factory premises, instead 
urging the government to declare the land a public green space or a 'deemed forest'.
- Community members have reclaimed Bogadi Road Junction and are demanding inclusive streets.

## Multi-turn Interaction


In [33]:
from langchain_core.messages import HumanMessage

while True:
    user_input = input("🧑 User: ")
    if user_input.lower() in {"exit", "quit"}:
        print("👋 Goodbye!")
        break

    print("🤖 AgentBot:")
    for chunk in agent.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config,
    ):
        # Check if 'agent' key and 'messages' exist
        if "agent" in chunk and "messages" in chunk["agent"]:
            for msg in chunk["agent"]["messages"]:
                print("--"*25)
                print("Agent Message:",msg.content)
        elif "tools" in chunk and "messages" in chunk["tools"]:
            for msg in chunk["tools"]["messages"]:
                print("--"*25)
                print("Tool Message:",msg.content)

🧑 User:  Hi


🤖 AgentBot:
--------------------------------------------------
Agent Message: You're welcome!


🧑 User:  Help me to learn about MCP and A2A protocols


🤖 AgentBot:
--------------------------------------------------
Agent Message: I will search for information about MCP and A2A protocols.
--------------------------------------------------
Tool Message: {"query": "MCP and A2A protocols", "follow_up_questions": null, "answer": null, "images": [], "results": [{"title": "MCP vs A2A: Understanding Context Protocols for AI Systems", "url": "https://devrelguide.com/blog/mcp-vs-a2a", "content": "As we've explored, the Model Context Protocol (MCP) and the Agent-to-Agent (A2A) protocol represent significant advancements in how we build intelligent, interconnected AI systems. MCP standardizes AI access to data and tools, while A2A aims to enable seamless collaboration between independent AI agents.\n\n### Key Takeaways\n\n### The Road Ahead\n\nAs MCP and A2A mature and gain adoption, we can anticipate:\n\n### Embracing MCP and A2A [...] The AI landscape is rapidly evolving, and with it, the need for clear standards to help different systems work 

🧑 User:  thanks for the info


🤖 AgentBot:
--------------------------------------------------
Agent Message: You're welcome!


🧑 User:  exit


👋 Goodbye!
